In [1]:
"""
Resume Matching Engine
Redrob AI Campus Hackathon — Individual Competition
"""

'\nResume Matching Engine\nRedrob AI Campus Hackathon — Individual Competition\n'

In [2]:
import math

In [3]:
SKILL_ALIASES = {
    # Languages
    "python": "python",           "pyhton": "python",
    "java": "java",
    "javascript": "javascript",   "javascrpit": "javascript",  "js": "javascript",
    "typescript": "typescript",   "typescrpit": "typescript",
    "c++": "cpp",                 "cpp": "cpp",
    "r": "r",
    "kotlin": "kotlin",
    # ML / Data
    "machinelearning": "machine_learning",
    "machine learning": "machine_learning",
    "ml": "machine_learning",
    "sklearn": "machine_learning",
    "deeplearning": "deep_learning",
    "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow",
    "pytorch": "pytorch",
    "keras": "keras",
    "nlp": "nlp",
    "bert": "bert",
    "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics",   "stats": "statistics",
    "regression": "regression",
    "clustering": "clustering",
    "data-viz": "data_visualization",
    "data visualization": "data_visualization",
    "data viz": "data_visualization",
    "matplotlib": "data_visualization",
    "tableau": "data_visualization",
    "power-bi": "data_visualization",
    "power bi": "data_visualization",
    "powerbi": "data_visualization",
    "pandas": "pandas",
    "numpy": "numpy",
    # Web — Frontend
    "react": "react",     "reacts": "react",    "reactjs": "react",
    "vue": "vue",         "vue.js": "vue",       "vuejs": "vue",
    "redux": "redux",
    "tailwind": "tailwind",
    "html/css": "html_css",  "html css": "html_css",
    "html": "html_css",      "css": "html_css",
    "jest": "jest",
    "graphql": "graphql",
    # Web — Backend
    "node.js": "nodejs",  "nodejs": "nodejs",  "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot",  "springboot": "spring_boot",
    "rest api": "rest_api",        "rest": "rest_api",  "restapi": "rest_api",
    "microservices": "microservices",
    # Databases
    "sql": "sql",
    "mysql": "mysql",      "mysq": "mysql",
    "postgresql": "postgresql",  "postgres": "postgresql",
    "mongodb": "mongodb",
    "redis": "redis",
    # DevOps / Cloud
    "docker": "docker",
    "kubernetes": "kubernetes",  "kubernates": "kubernetes",  "k8s": "kubernetes",
    "ci/cd": "ci_cd",            "cicd": "ci_cd",             "ci cd": "ci_cd",
    "aws": "aws",
    # Mobile
    "android": "android",
    "firebase": "firebase",
    # CS Fundamentals
    "algorithms": "algorithms",   "algoritms": "algorithms",
    "data structure": "data_structures",  "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    # Design
    "ui/ux": "ui_ux",  "ui ux": "ui_ux",
    "figma": "figma",
}

In [4]:
#Datasets
RESUMES = [
    ("Arjun Sharma",    "Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning"),
    ("Priya Nair",      "JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS"),
    ("Rahul Gupta",     "Java, Spring Boot, MySql, Microservices, Docker, kubernates"),
    ("Sneha Patel",     "Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib"),
    ("Vikram Singh",    "C++, Algoritms, Data Structure, competitive programming, python"),
    ("Ananya Krishnan", "javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD"),
    ("Karan Mehta",     "Python, Sklearn, XGboost, feature engineering, SQL, tableau"),
    ("Deepika Rao",     "Java, Android, Kotlin, Firebase, REST, UI/UX, figma"),
    ("Aditya Kumar",    "Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest"),
    ("Meera Iyer",      "python, R, statistics, ML, regression, clustering, Power-BI"),
]
 
JDS = [
    ("JD-1", "Kakao",  "ML Engineer",
     "Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, "
     "Data Visualization, NLP, BERT, Feature Engineering, Statistics"),
    ("JD-2", "Naver",  "Backend Engineer",
     "Java, Spring Boot, MySQL, PostgreSQL, Microservices, Docker, "
     "Kubernetes, REST API, CI/CD, Redis"),
    ("JD-3", "Line",   "Frontend Engineer",
     "JavaScript, React, Vue, TypeScript, REST API, HTML/CSS, "
     "Node.js, GraphQL, Redux, Jest, AWS"),
]

In [5]:
# STEP 1 & 2: NORMALIZE + DEDUPLICATE SKILLS

def normalize(raw_skills_str):
    tokens = [t.strip().lower() for t in raw_skills_str.split(',')]
    seen, result = set(), []
    for token in tokens:
        if token in SKILL_ALIASES:
            canonical = SKILL_ALIASES[token]
            if canonical not in seen:
                seen.add(canonical)
                result.append(canonical)
    return result
 
 
normalized_resumes = [(name, normalize(raw)) for name, raw in RESUMES]

In [6]:
# STEP 3: BUILD VOCABULARY
# Sorted alphabetically from all normalized resume skills.
vocab_set = set()
for _, skills in normalized_resumes:
    vocab_set.update(skills)
vocab = sorted(vocab_set)
vocab_index = {skill: idx for idx, skill in enumerate(vocab)}
V = len(vocab)

In [7]:
# STEP 4: COMPUTE TF-IDF VECTORS FOR RESUMES
# Document frequency
df = [0] * V
for _, skills in normalized_resumes:
    for skill in skills:
        df[vocab_index[skill]] += 1
 
# IDF — natural log, no smoothing, corpus size = 10
idf = [math.log(10.0 / df[i]) for i in range(V)]
 
# TF-IDF vectors
tfidf_vectors = []
for name, skills in normalized_resumes:
    N = len(skills)
    vec = [0.0] * V
    for skill in skills:
        idx = vocab_index[skill]
        vec[idx] = (1.0 / N) * idf[idx]
    tfidf_vectors.append((name, vec))

In [8]:
# STEP 5: BUILD JD BINARY VECTORS
def build_jd_vector(raw_skills_str):
    skills = normalize(raw_skills_str)
    vec = [0.0] * V
    for skill in skills:
        if skill in vocab_index:
            vec[vocab_index[skill]] = 1.0
    return vec
 
 
jd_vectors = [
    (jd_id, company, role, build_jd_vector(raw))
    for jd_id, company, role, raw in JDS
]

In [9]:
# Validation: show matching skills per JD before final ranking
for jd_id, company, role, jd_vec in jd_vectors:
    print(f"\n{jd_id} active vocab skills: {[vocab[i] for i in range(V) if jd_vec[i]]}")
    for name, rvec in tfidf_vectors:
        overlap = [vocab[i] for i in range(V) if rvec[i] > 0 and jd_vec[i] > 0]
        if overlap:
            print(f"  {name} overlaps: {overlap}")


JD-1 active vocab skills: ['bert', 'data_visualization', 'deep_learning', 'feature_engineering', 'machine_learning', 'nlp', 'python', 'sql', 'statistics', 'tensorflow']
  Arjun Sharma overlaps: ['deep_learning', 'machine_learning', 'python', 'sql']
  Sneha Patel overlaps: ['bert', 'data_visualization', 'nlp', 'python', 'tensorflow']
  Vikram Singh overlaps: ['python']
  Ananya Krishnan overlaps: ['python']
  Karan Mehta overlaps: ['data_visualization', 'feature_engineering', 'machine_learning', 'python', 'sql']
  Meera Iyer overlaps: ['data_visualization', 'machine_learning', 'python', 'statistics']

JD-2 active vocab skills: ['ci_cd', 'docker', 'java', 'kubernetes', 'microservices', 'mysql', 'postgresql', 'rest_api', 'spring_boot']
  Priya Nair overlaps: ['rest_api']
  Rahul Gupta overlaps: ['docker', 'java', 'kubernetes', 'microservices', 'mysql', 'spring_boot']
  Ananya Krishnan overlaps: ['ci_cd', 'postgresql']
  Deepika Rao overlaps: ['java', 'rest_api']

JD-3 active vocab skills

In [10]:
# STEP 6: COSINE SIMILARITY & TOP-3 RANKING
#   Cosine(A, B) = (A · B) / (|A| × |B|)
#   Ties broken alphabetically by candidate name.

def cosine_sim(a, b):
    dot    = sum(a[i] * b[i] for i in range(V))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)

In [11]:
# OUTPUT
print("=" * 60)
print("       RESUME MATCHING ENGINE — FINAL RESULTS")
print("=" * 60)
 
for jd_id, company, role, jd_vec in jd_vectors:
    scores = [
        (name, cosine_sim(rvec, jd_vec))
        for name, rvec in tfidf_vectors
    ]
    # Sort: descending score; alphabetical name on tie
    scores.sort(key=lambda x: (-round(x[1], 2), x[0]))
    top3 = scores[:3]
 
    print(f"\n{jd_id} — {company} ({role})")
    print(", ".join(f"{name}({score:.2f})" for name, score in top3))
 
print("\n" + "=" * 60)

       RESUME MATCHING ENGINE — FINAL RESULTS

JD-1 — Kakao (ML Engineer)
Sneha Patel(0.57), Karan Mehta(0.53), Arjun Sharma(0.40)

JD-2 — Naver (Backend Engineer)
Rahul Gupta(0.81), Ananya Krishnan(0.28), Deepika Rao(0.19)

JD-3 — Line (Frontend Engineer)
Aditya Kumar(0.67), Priya Nair(0.58), Ananya Krishnan(0.35)

